插值后再进行拟合容易出现过拟合现象，除非样本过少可以通过插值丰富样本内容后再拟合，其余情况1不推荐。
替代方式：正则化、岭回归

数据准备

In [1]:
import numpy as np
import pandas as pd

# 读取Excel数据
data = pd.read_excel('插值整合后的数据.xlsx')

# 提取THz值和吸收强度
x = data.iloc[:, 0].values.reshape(-1, 1)  # THz值

# 提取吸收强度（每个化合物的吸收强度在1-201列之间）
num_compounds = 20
train_y = data.iloc[:, 1:201].values  # 除了THz值的其他数据，下面的代码就是取出测试集数据
test_y = data.iloc[:, 10::10].values  # 测试集（每10列取一次，最后一列）

In [3]:
train_y.shape

(1630, 200)

In [6]:
# 构建要排除的列索引
exclude_indices = list(range(9, 200, 10))  # 生成[9, 19, 29, ..., 199]

# 生成所有列的索引
all_indices = np.arange(train_y.shape[1])  # 所有列的索引 [0, 1, 2, ..., 200]

# 选择不在 exclude_indices 中的列
selected_indices = np.setdiff1d(all_indices, exclude_indices)

# 取出选中的列
filtered_train_y = train_y[:, selected_indices]

print(filtered_train_y.shape)  # 打印新数组的形状

(1630, 180)


In [7]:
train_y = filtered_train_y

In [8]:
train_y.shape

(1630, 180)

训练SVR模型

In [14]:
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 创建 SVR 模型
models = []
for i in range(num_compounds):
    svr = make_pipeline(StandardScaler(), SVR(kernel='rbf'))
    svr.fit(x, train_y[:, i*9])
    models.append(svr)

预测

In [15]:
predictions = []
for model in models:
    pred = model.predict(x)  # 直接用所有 THz 值进行预测
    predictions.append(pred)

predictions = np.array(predictions).T  # 转置以匹配原始格式

评估模型

In [16]:
from sklearn.metrics import mean_squared_error

mse = [mean_squared_error(test_y[:, i], predictions[:, i]) for i in range(num_compounds)]
print(f'每一个化合物的MSE分别为: {mse}')

每一个化合物的MSE分别为: [258.9535416920827, 161.82448257910858, 61.12880307177368, 144.642230991835, 40.11748337561585, 97.35010676745375, 118.36197585511117, 0.023901835180073442, 781.9279682699718, 838.3437159034152, 730.5385157640898, 619.6750982480794, 0.0029937707867734088, 0.0028929931243468977, 0.0036324888142345232, 0.00336878143749441, 0.007256775619935502, 0.015558769128035454, 0.009238086383833955, 0.04912828835816372]


In [17]:
from sklearn.metrics import r2_score

# test_y 是真实值，predictions是模型预测值
r2_scores = [r2_score(test_y[:, i], predictions[:, i]) for i in range(num_compounds)]
print(f'每个化合物的R方值为: {r2_scores}')

每个化合物的R方值为: [0.7638065075446807, 0.8186514257285671, 0.8567361230785613, 0.9334491916852942, 0.926927263183251, 0.869925418040558, 0.8876849106152785, 0.7691529608420271, 0.7506542510616513, 0.7575630527420789, 0.705983520089516, 0.8718587268644976, 0.7895757343932196, 0.42724626382987885, 0.4236608205586493, 0.8620805217963131, 0.22052267893325583, 0.09142754989377178, 0.3194306836031984, -0.12309686377997697]
